# Seattle Airbnb — Data Understanding, Integration & EDA

**DSBA 6156 | Fall 2026 | Dzmitry Prybysh**

This is our reproducible working notebook for understanding the five input datasets, validating how they relate, building analysis-ready tables, and exploring initial patterns.

## Notebook Roadmap
1. **Setup**
2. **Data Inventory**
3. **Source Data Exploration**
   - 3.1 Listings — `listings.csv`
   - 3.2 Calendar — `calendar.csv`
   - 3.3 Reviews — `reviews.csv`
   - 3.4 Walk Score — `seattle_walkscore_raw.csv`
   - 3.5 Holidays & Events — `seattle_holidays_events_2016.csv`
4. **Data Integration** *(upcoming)*
5. **Data Cleaning & Preparation** *(upcoming)*
6. **Exploratory Data Analysis (EDA)** *(upcoming)*
7. **Summary & Handoff** *(upcoming)*

**Important:** The Airbnb calendar's `available` flag does not directly identify bookings. Derived occupancy or revenue should be described as proxies, not verified observations.


## 1. Setup

Load project dependencies and configure data paths. The input datasets are read directly from `data/` in a read-only, non-destructive manner.


In [1]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)

# Works when Jupyter starts in the repository root or the notebooks/ directory.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data'
assert DATA_DIR.is_dir(), f'Data directory not found: {DATA_DIR}'
print('Project root:', PROJECT_ROOT)


Project root: /Users/Dzmitry/UNC 2026 AI program/Academics/Machine Learning/Project airbnb Seattle/seattle-airbnb-ml


## 2. Data Inventory

These inputs represent **different units of observation**. First check which files are available; do not merge them yet.


In [2]:
source_files = [
    'listings.csv',
    'calendar.csv',
    'reviews.csv',
    'seattle_holidays_events_2016.csv',
    'seattle_walkscore_raw.csv',
]

inventory = pd.DataFrame([
    {
        'file': name,
        'exists': (DATA_DIR / name).is_file(),
        'size_MB': round((DATA_DIR / name).stat().st_size / 1_000_000, 2)
                   if (DATA_DIR / name).is_file() else None,
    }
    for name in source_files
])
display(inventory)


,file,exists,size_MB
0,listings.csv,True,16.87
1,calendar.csv,True,36.65
2,reviews.csv,True,36.59
3,seattle_holidays_events_2016.csv,True,0.00
4,seattle_walkscore_raw.csv,True,0.01


## 3. Source Data Exploration

In this section, we examine each of the five source datasets individually before joining or transforming them.

### 3.1 Listings — `listings.csv`

#### 3.1.1 Data Exploration

Questions to answer: How many listings and columns are present? What does one row represent? Are listing IDs unique? Which features contain missing values? Are apparent numbers stored as strings?


In [3]:
listings_path = DATA_DIR / 'listings.csv'
if not listings_path.is_file():
    raise FileNotFoundError(
        f'{listings_path} is missing. Copy listings.csv from the team Google Drive data folder.'
    )

listings = pd.read_csv(listings_path, low_memory=False)
print('Shape (rows, columns):', listings.shape)
display(listings.head())


Shape (rows, columns): (3818, 92)


,id,listing_url,scrape_id,last_scraped,name,summary,space,description,experiences_offered,neighborhood_overview,notes,transit,thumbnail_url,medium_url,picture_url,xl_picture_url,host_id,host_url,host_name,host_since,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,street,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,city,state,zipcode,market,smart_location,country_code,country,latitude,longitude,is_location_exact,property_type,room_type,accommodates,bathrooms,bedrooms,beds,bed_type,amenities,square_feet,price,weekly_price,monthly_price,security_deposit,cleaning_fee,guests_included,extra_people,minimum_nights,maximum_nights,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,requires_license,license,jurisdiction_names,instant_bookable,cancellation_policy,require_guest_profile_picture,require_guest_phone_verification,calculated_host_listings_count,reviews_per_month
0,241032,https://www.airbnb.com/rooms/241032,20160104002432,2016-01-04,Stylish Queen Anne Apartment,NaN,Make your self at home in this charming one-be...,Make your self at home in this charming one-be...,none,NaN,NaN,NaN,NaN,NaN,https://a1.muscache.com/ac/pictures/67560560/c...,NaN,956883,https://www.airbnb.com/users/show/956883,Maija,2011-08-11,"Seattle, Washington, United States","I am an artist, interior designer, and run a s...",within a few hours,96%,100%,f,https://a0.muscache.com/ac/users/956883/profil...,https://a0.muscache.com/ac/users/956883/profil...,Queen Anne,3.0,3.0,"['email', 'phone', 'reviews', 'kba']",t,t,"Gilman Dr W, Seattle, WA 98119, United States",Queen Anne,West Queen Anne,Queen Anne,Seattle,WA,98119,Seattle,"Seattle, WA",US,United States,47.636289,-122.371025,t,Apartment,Entire home/apt,4,1.0,1.0,1.0,Real Bed,"{TV,""Cable TV"",Internet,""Wireless Internet"",""A...",NaN,$85.00,NaN,NaN,NaN,NaN,2,$5.00,1,365,4 weeks ago,t,14,41,71,346,2016-01-04,207,2011-11-01,2016-01-02,95.0,10.0,10.0,10.0,10.0,9.0,10.0,f,NaN,WASHINGTON,f,moderate,f,f,2,4.07
1,953595,https://www.airbnb.com/rooms/953595,20160104002432,2016-01-04,Bright & Airy Queen Anne Apartment,Chemically sensitive? We've removed the irrita...,"Beautiful, hypoallergenic apartment in an extr...",Chemically sensitive? We've removed the irrita...,none,"Queen Anne is a wonderful, truly functional vi...",What's up with the free pillows? Our home was...,"Convenient bus stops are just down the block, ...",https://a0.muscache.com/ac/pictures/14409893/f...,https://a0.muscache.com/im/pictures/14409893/f...,https://a0.muscache.com/ac/pictures/14409893/f...,https://a0.muscache.com/ac/pictures/14409893/f...,5177328,https://www.airbnb.com/users/show/5177328,Andrea,2013-02-21,"Seattle, Washington, United States",Living east coast/left coast/overseas. Time i...,within an hour,98%,100%,t,https://a0.muscache.com/ac/users/5177328/profi...,https://a0.muscache.com/ac/users/5177328/profi...,Queen Anne,6.0,6.0,"['email', 'phone', 'facebook', 'linkedin', 're...",t,t,"7th Avenue West, Seattle, WA 98119, United States",Queen Anne,West Queen Anne,Queen Anne,Seattle,WA,98119,Seattle,"Seattle, WA",US,United States,47.639123,-122.365666,t,Apartment,Entire home/apt,4,1.0,1.0,1.0,Real Bed,"{TV,Internet,""Wireless Internet"",Kitchen,""Free...",NaN,$150.00,"$1,000.00","$3,000.00",$100.00,$40.00,1,$0.00,2,90,today,t,13,13,16,291,2016-01-04,43,2013-08-19,2015-12-29,96.0,10.0,10.0,10.0,10.0,10.0,10.0,f,NaN,WASHINGTON,f,strict,t,t,6,1.48
2,3308979,https://www.airbnb.com/rooms/3308979,20160104002432,2016-01-04,New Modern House-Amazi

In [4]:
print('Column types:')
print(listings.dtypes.to_string())

print('\nMissing values by column (top 30):')
missing_summary = pd.DataFrame({
    'missing_count': listings.isna().sum(),
    'missing_pct': (listings.isna().mean() * 100).round(2),
}).sort_values(['missing_count', 'missing_pct'], ascending=False)
display(missing_summary.head(30))

print('\nDuplicate full rows:', listings.duplicated().sum())
if 'id' in listings.columns:
    print('Duplicate listing IDs:', listings['id'].duplicated().sum())
    print('Unique listing IDs:', listings['id'].nunique(dropna=True))


Column types:
id                                    int64
listing_url                          object
scrape_id                             int64
last_scraped                         object
name                                 object
summary                              object
space                                object
description                          object
experiences_offered                  object
neighborhood_overview                object
notes                                object
transit                              object
thumbnail_url                        object
medium_url                           object
picture_url                          object
xl_picture_url                       object
host_id                               int64
host_url                             object
host_name                            object
host_since                           object
host_location                        object
host_about                           object
host_response_time

,missing_count,missing_pct
license,3818,100.00
square_feet,3721,97.46
monthly_price,2301,60.27
security_deposit,1952,51.13
weekly_price,1809,47.38
notes,1606,42.06
neighborhood_overview,1032,27.03
cleaning_fee,1030,26.98
transit,934,24.46
host_about,859,22.50



Duplicate full rows: 0
Duplicate listing IDs: 0
Unique listing IDs: 3818


In [5]:
# Verification of scrape date, neighborhood completeness, and review rating missingness
print('1. Scrape date distribution across all listings:')
display(listings['last_scraped'].value_counts(dropna=False).to_frame('count'))

print('\n2. Missing values in neighborhood taxonomy columns:')
hood_missing = listings[['neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed']].isna().sum().to_frame('missing_count')
hood_missing['missing_pct'] = (listings[['neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed']].isna().mean() * 100).round(2)
display(hood_missing)

print('\n3. Missing review scores breakdown: zero reviews (N=627) vs with reviews (N=3,191):')
review_fields_all = [
    'first_review', 'last_review', 'reviews_per_month',
    'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness',
    'review_scores_checkin', 'review_scores_communication', 'review_scores_location',
    'review_scores_value'
]
zero_reviews_mask = (listings['number_of_reviews'] == 0)
has_reviews_mask = (listings['number_of_reviews'] > 0)

review_missing_table = pd.DataFrame({
    'zero_reviews_missing (N=627)': listings.loc[zero_reviews_mask, review_fields_all].isna().sum(),
    'has_reviews_missing (N=3,191)': listings.loc[has_reviews_mask, review_fields_all].isna().sum(),
    'total_missing (N=3,818)': listings[review_fields_all].isna().sum(),
})
display(review_missing_table)

print('\n4. Data loss under different listwise deletion criteria for review ratings:')
rating_fields = [
    'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness',
    'review_scores_checkin', 'review_scores_communication', 'review_scores_location',
    'review_scores_value'
]
zero_reviews_loss = (listings['number_of_reviews'] == 0).sum()
missing_rating_loss = listings['review_scores_rating'].isna().sum()
missing_any_rating_loss = listings[rating_fields].isna().any(axis=1).sum()
n_listings = len(listings)

deletion_impact = pd.DataFrame([
    {
        'Condition': 'Zero reviews (number_of_reviews == 0)',
        'Listings Removed': zero_reviews_loss,
        'Share of Total': f'{(zero_reviews_loss / n_listings) * 100:.2f}%'
    },
    {
        'Condition': 'Missing overall rating (review_scores_rating is NaN)',
        'Listings Removed': missing_rating_loss,
        'Share of Total': f'{(missing_rating_loss / n_listings) * 100:.2f}%'
    },
    {
        'Condition': 'Missing any of 7 rating categories',
        'Listings Removed': missing_any_rating_loss,
        'Share of Total': f'{(missing_any_rating_loss / n_listings) * 100:.2f}%'
    },
])
display(deletion_impact)


1. Scrape date distribution across all listings:


,count
last_scraped,
2016-01-04,3818



2. Missing values in neighborhood taxonomy columns:


,missing_count,missing_pct
neighbourhood,416,10.9
neighbourhood_cleansed,0,0.0
neighbourhood_group_cleansed,0,0.0



3. Missing review scores breakdown: zero reviews (N=627) vs with reviews (N=3,191):


,zero_reviews_missing (N=627),"has_reviews_missing (N=3,191)","total_missing (N=3,818)"
first_review,627,0,627
last_review,627,0,627
reviews_per_month,627,0,627
review_scores_rating,627,20,647
review_scores_accuracy,627,31,658
review_scores_cleanliness,627,26,653
review_scores_checkin,627,31,658
review_scores_communication,627,24,651
review_scores_location,627,28,655
review_scores_value,627,29,656



4. Data loss under different listwise deletion criteria for review ratings:


,Condition,Listings Removed,Share of Total
0,Zero reviews (number_of_reviews == 0),627,16.42%
1,Missing overall rating (review_scores_rating i...,647,16.95%
2,Missing any of 7 rating categories,660,17.29%


#### 3.1.2 Preliminary Findings

All observations below are strictly verified by executed code on `listings.csv`:

1. **Dataset shape and row grain:** `3,818` rows and `92` columns. Every listing has `last_scraped == '2016-01-04'` (`3,818` out of `3,818`, `0` NaNs), confirming a single-day snapshot. Each row represents a single unique Airbnb listing in Seattle (`3,818` unique `id` values, `0` duplicate full rows, `0` duplicate listing IDs).
2. **Data quality and missing values:**
   - `license` (`100.00%` missing) and `square_feet` (`97.46%` missing) are near-empty.
   - Optional pricing and deposit columns exhibit high missingness (`monthly_price` `60.27%`, `security_deposit` `51.13%`, `weekly_price` `47.38%`, `cleaning_fee` `26.98%`).
   - Raw `neighbourhood` is missing in `416` rows (`10.90%`), whereas `neighbourhood_cleansed` and `neighbourhood_group_cleansed` are `100.00%` complete across all `3,818` listings (`0` missing values).
3. **Structure of missing review fields and listwise deletion impact:**
   - For all `627` listings with `0` reviews, dates (`first_review`, `last_review`), `reviews_per_month`, and all seven rating score fields are missing (`100%`).
   - For the `3,191` listings with reviews, dates and `reviews_per_month` are `100%` complete, but missing values remain across rating categories: `review_scores_rating` (`20`), `review_scores_communication` (`24`), `review_scores_cleanliness` (`26`), `review_scores_location` (`28`), `review_scores_value` (`29`), `review_scores_accuracy` (`31`), and `review_scores_checkin` (`31`). The majority of missing rating scores occur among listings with zero reviews; additional missing values occur among listings with reviews.
   - Under listwise deletion, data loss depends on the required review fields: dropping listings with zero reviews removes `627` listings (`16.42%`), dropping listings missing the overall `review_scores_rating` removes `647` listings (`16.95%`), and dropping listings missing any of the seven rating categories removes `660` listings (`17.29%`).
4. **Columns needing special parsing:**
   - Currency strings (`$` and `,` → `float`): `price`, `weekly_price`, `monthly_price`, `security_deposit`, `cleaning_fee`, `extra_people`.
   - Percentage strings (`%` → `float`): `host_response_rate`, `host_acceptance_rate`.
   - Boolean flags (`'t'`/`'f'` → `bool`/`0-1`): `host_is_superhost`, `host_has_profile_pic`, `host_identity_verified`, `is_location_exact`, `has_availability`, `requires_license`, `instant_bookable`, `require_guest_profile_picture`, `require_guest_phone_verification`.
   - Date strings (`object` → `datetime`): `last_scraped`, `host_since`, `calendar_last_scraped`, `first_review`, `last_review`.
   - Delimited lists/sets: `amenities`, `host_verifications`.

##### Analytical Considerations for Next Steps
- **Primary key:** Maintain `id` as the unique identifier for listings (`3,818` rows).
- **Feature selection candidates:** Flag `license` and `square_feet` as candidates for removal during data cleaning due to near-complete missingness.
- **Taxonomy key:** Use `neighbourhood_cleansed` (complete across all `3,818` listings) rather than raw `neighbourhood` when integrating geographic features.
- **Aggregation strategy:** Plan to aggregate multi-record datasets (`calendar.csv` and `reviews.csv`) to the listing level before merging into `listings.csv` to avoid expanding the row count.


### 3.2 Calendar — `calendar.csv`

#### 3.2.1 Data Exploration

In this section, we examine `calendar.csv` to understand its volume, temporal coverage, missing values, duplicates, and relationship to `listings.csv` before performing any joins or transformations.

**Guiding principles for this step:**
- Analyze the raw table as-is (no cleaning, dropping, or merging).
- Treat `available == 'f'` strictly as an availability state (an occupancy proxy), not a verified booking.


In [6]:
# 1-3. Load calendar.csv, inspect shape, column types, and first rows
calendar_path = DATA_DIR / 'calendar.csv'
assert calendar_path.is_file(), f'calendar.csv not found at {calendar_path}'

# Load with low_memory=False; parse dates if needed or inspect raw types first
calendar = pd.read_csv(calendar_path, low_memory=False)

print('1. Dataset shape (rows, columns):', calendar.shape)
print('\n2. Column names and data types:')
display(calendar.dtypes.astype(str).rename('dtype').to_frame())

print('\n3. First 5 rows:')
display(calendar.head())


1. Dataset shape (rows, columns): (1393570, 4)

2. Column names and data types:


,dtype
listing_id,int64
date,object
available,object
price,object



3. First 5 rows:


,listing_id,date,available,price
0,241032,2016-01-04,t,$85.00
1,241032,2016-01-05,t,$85.00
2,241032,2016-01-06,f,NaN
3,241032,2016-01-07,f,NaN
4,241032,2016-01-08,f,NaN


In [7]:
# 4-8. Missing values, duplicates, unique IDs, date range, and record counts
print('4. Missing values (count and percentage):')
calendar_missing = pd.DataFrame({
    'missing_count': calendar.isna().sum(),
    'missing_pct': (calendar.isna().mean() * 100).round(2),
})
display(calendar_missing)

print('\n5. Duplicate checks:')
full_dupes = calendar.duplicated().sum()
pair_dupes = calendar.duplicated(subset=['listing_id', 'date']).sum()
print(f'Duplicate full rows: {full_dupes}')
print(f'Duplicate (listing_id, date) pairs: {pair_dupes}')

print('\n6. Unique listings count:')
unique_listings_cal = calendar['listing_id'].nunique(dropna=True)
print(f'Number of unique listing_id values in calendar: {unique_listings_cal}')

print('\n7. Date range and coverage:')
min_date = calendar['date'].min()
max_date = calendar['date'].max()
n_unique_dates = calendar['date'].nunique()
print(f'Minimum date: {min_date}')
print(f'Maximum date: {max_date}')
print(f'Number of unique dates: {n_unique_dates}')

print('\n8. Records per listing:')
records_per_listing = calendar.groupby('listing_id').size()
print(f'Min records per listing: {records_per_listing.min()}')
print(f'Max records per listing: {records_per_listing.max()}')
print('Distribution of record count per listing:')
display(records_per_listing.value_counts().rename('listing_count').to_frame())


4. Missing values (count and percentage):


,missing_count,missing_pct
listing_id,0,0.00
date,0,0.00
available,0,0.00
price,459028,32.94



5. Duplicate checks:
Duplicate full rows: 0
Duplicate (listing_id, date) pairs: 0

6. Unique listings count:
Number of unique listing_id values in calendar: 3818

7. Date range and coverage:
Minimum date: 2016-01-04
Maximum date: 2017-01-02
Number of unique dates: 365

8. Records per listing:
Min records per listing: 365
Max records per listing: 365
Distribution of record count per listing:


,listing_count
365,3818


In [8]:
# Verification of date parsing and unbroken sequence coverage
parsed_cal_dates = pd.to_datetime(calendar['date'], format='%Y-%m-%d', errors='coerce')
date_parse_errors = parsed_cal_dates.isna().sum()
min_cal_date = parsed_cal_dates.min()
max_cal_date = parsed_cal_dates.max()
expected_cal_range = pd.date_range(start=min_cal_date, end=max_cal_date, freq='D')
actual_unique_cal_dates = parsed_cal_dates.dropna().unique()
missing_cal_days = set(expected_cal_range) - set(actual_unique_cal_dates)

print('Calendar date parsing and sequence verification:')
print(f'Date parsing errors: {date_parse_errors}')
print(f'Observed minimum date: {min_cal_date.strftime("%Y-%m-%d")}')
print(f'Observed maximum date: {max_cal_date.strftime("%Y-%m-%d")}')
print(f'Duration of observed range (days): {len(expected_cal_range)}')
print(f'Unique dates present: {len(actual_unique_cal_dates)}')
print(f'Missing days within range: {len(missing_cal_days)}')
print(f'Unbroken daily sequence confirmed: {len(missing_cal_days) == 0 and date_parse_errors == 0}')


Calendar date parsing and sequence verification:
Date parsing errors: 0
Observed minimum date: 2016-01-04
Observed maximum date: 2017-01-02
Duration of observed range (days): 365
Unique dates present: 365
Missing days within range: 0
Unbroken daily sequence confirmed: True


In [9]:
# 9-11. Available distribution, ID matching with listings.csv, and row grain
print('9. Distribution of the available field (t/f):')
avail_summary = pd.DataFrame({
    'count': calendar['available'].value_counts(dropna=False),
    'pct': (calendar['available'].value_counts(normalize=True, dropna=False) * 100).round(2),
})
display(avail_summary)

# Cross-check price missingness vs availability
null_price_when_f = calendar.loc[calendar['available'] == 'f', 'price'].isna().sum()
null_price_when_t = calendar.loc[calendar['available'] == 't', 'price'].isna().sum()
print(f'Missing prices when available == "f": {null_price_when_f} (out of {avail_summary.loc["f", "count"]})')
print(f'Missing prices when available == "t": {null_price_when_t} (out of {avail_summary.loc["t", "count"]})')

print('\n10. Matching listing_id with listings.csv:')
listings_ids_set = set(listings['id'])
calendar_ids_set = set(calendar['listing_id'])
ids_match_exactly = (listings_ids_set == calendar_ids_set)
diff_cal_minus_list = len(calendar_ids_set - listings_ids_set)
diff_list_minus_cal = len(listings_ids_set - calendar_ids_set)

print(f'Unique listings in listings.csv: {len(listings_ids_set)}')
print(f'Unique listings in calendar.csv: {len(calendar_ids_set)}')
print(f'Exact 1-to-1 set match: {ids_match_exactly}')
print(f'Listings in calendar but not in listings.csv: {diff_cal_minus_list}')
print(f'Listings in listings.csv but not in calendar: {diff_list_minus_cal}')

print('\n11. Row grain & join keys:')
is_composite_key = (len(calendar) == calendar[['listing_id', 'date']].drop_duplicates().shape[0])
print(f'Does (listing_id, date) uniquely identify each row? {is_composite_key}')
print('Identified row grain: 1 row represents a single listing on a specific calendar date (listing-day grain).')
print('Join keys for future steps: listing_id (to join with listings.csv) and date (to join with holidays/events).')


9. Distribution of the available field (t/f):


,count,pct
available,,
t,934542,67.06
f,459028,32.94


Missing prices when available == "f": 459028 (out of 459028)
Missing prices when available == "t": 0 (out of 934542)

10. Matching listing_id with listings.csv:
Unique listings in listings.csv: 3818
Unique listings in calendar.csv: 3818
Exact 1-to-1 set match: True
Listings in calendar but not in listings.csv: 0
Listings in listings.csv but not in calendar: 0

11. Row grain & join keys:
Does (listing_id, date) uniquely identify each row? True
Identified row grain: 1 row represents a single listing on a specific calendar date (listing-day level).
Join keys for future steps: listing_id (to join with listings.csv) and date (to join with holidays/events).


#### 3.2.2 Preliminary Findings

All observations below are strictly verified by executed code on `calendar.csv`:

1. **Dataset shape and structure:** `calendar.csv` contains `1,393,570` rows and `4` columns (`listing_id`, `date`, `available`, `price`).
2. **Data types:** `listing_id` is `int64`, while `date`, `available`, and `price` are stored as `object` (strings). `price` includes currency formatting (`$` and commas).
3. **Missing values:**
   - `listing_id`, `date`, and `available` have `0` missing values (`0.00%`).
   - `price` has `459,028` missing values (`32.94%`), matching exactly the `459,028` records where `available == 'f'`. When a listing is marked unavailable, no price is recorded in the calendar.
4. **Duplicates and uniqueness:** `0` duplicate full rows. The pair `(listing_id, date)` is strictly unique across all `1,393,570` records (`0` duplicates).
5. **Temporal coverage and balanced panel:**
   - Date parsing succeeded with `0` errors across all rows.
   - Dates span from `2016-01-04` through `2017-01-02` with `0` missing days within the 365-day duration, confirming an unbroken daily sequence.
   - Exactly `3,818` unique `listing_id` values are present, each with exactly `365` daily records (`3,818 × 365 = 1,393,570`), forming a complete, balanced panel.
6. **Available distribution:**
   - `available == 't'`: `934,542` days (`67.06%`).
   - `available == 'f'`: `459,028` days (`32.94%`).
   - *Important constraint:* Dates marked `available == 'f'` represent unavailable calendar days and serve as an **occupancy proxy**, but should not be interpreted as confirmed guest bookings (a host may block dates for personal use or maintenance).
7. **Cross-table alignment:** `calendar.csv` and `listings.csv` match perfectly on `listing_id` (`3,818` in both, `0` orphaned records in either direction).

##### Analytical Considerations for Next Steps
- **Row grain:** One record per listing per date (listing-day level).
- **Aggregation strategy:** To merge calendar signals into the listing-level table without expanding rows, metrics (e.g., share of unavailable days (not verified occupancy), price variance, peak season availability) must be aggregated by `listing_id`.
- **Candidate event linkage:** `date` serves as a candidate key for future integration with the holidays and events dataset, subject to empirical verification of event dates.


### 3.3 Reviews — `reviews.csv`

#### 3.3.1 Data Exploration

In this section, we examine `reviews.csv` to understand its structure, missing values, duplicates, temporal coverage, reviewer characteristics, and alignment with `listings.csv` before performing any feature extraction or text processing.


In [10]:
# 1-3. Load reviews.csv, inspect shape, column types, and first rows
reviews_path = DATA_DIR / 'reviews.csv'
assert reviews_path.is_file(), f'reviews.csv not found at {reviews_path}'

reviews = pd.read_csv(reviews_path, low_memory=False)

print('1. Dataset shape (rows, columns):', reviews.shape)
print('\n2. Column names and data types:')
display(reviews.dtypes.astype(str).rename('dtype').to_frame())

print('\n3. First 5 rows:')
display(reviews.head())


1. Dataset shape (rows, columns): (84849, 6)

2. Column names and data types:


,dtype
listing_id,int64
id,int64
date,object
reviewer_id,int64
reviewer_name,object
comments,object



3. First 5 rows:


,listing_id,id,date,reviewer_id,reviewer_name,comments
0,7202016,38917982,2015-07-19,28943674,Bianca,Cute and cozy place. Perfect location to every...
1,7202016,39087409,2015-07-20,32440555,Frank,Kelly has a great room in a very central locat...
2,7202016,39820030,2015-07-26,37722850,Ian,"Very spacious apartment, and in a great neighb..."
3,7202016,40813543,2015-08-02,33671805,George,Close to Seattle Center and all it has to offe...
4,7202016,41986501,2015-08-10,34959538,Ming,Kelly was a great host and very accommodating ...


In [11]:
# 4-6. Missing values, duplicates, and unique identifiers
print('4. Missing values (count and percentage):')
reviews_missing = pd.DataFrame({
    'missing_count': reviews.isna().sum(),
    'missing_pct': (reviews.isna().mean() * 100).round(2),
})
display(reviews_missing)

# Check for whitespace-only strings in comments
empty_comments = ((reviews['comments'].str.strip() == '') & reviews['comments'].notna()).sum()
print(f'Empty whitespace-only comments: {empty_comments}')

print('\n5. Duplicate checks:')
full_dupes = reviews.duplicated().sum()
id_dupes = reviews['id'].duplicated().sum()
triplet_dupes = reviews.duplicated(subset=['listing_id', 'reviewer_id', 'date']).sum()
print(f'Duplicate full rows: {full_dupes}')
print(f'Duplicate review IDs: {id_dupes}')
print(f'Duplicate (listing_id, reviewer_id, date) triplets: {triplet_dupes}')

print('\n6. Unique identifier counts:')
print(f'Unique review IDs (id): {reviews["id"].nunique(dropna=True)}')
print(f'Unique listings with reviews (listing_id): {reviews["listing_id"].nunique(dropna=True)}')
print(f'Unique reviewers (reviewer_id): {reviews["reviewer_id"].nunique(dropna=True)}')


4. Missing values (count and percentage):


,missing_count,missing_pct
listing_id,0,0.00
id,0,0.00
date,0,0.00
reviewer_id,0,0.00
reviewer_name,0,0.00
comments,18,0.02


Empty whitespace-only comments: 0

5. Duplicate checks:
Duplicate full rows: 0
Duplicate review IDs: 0
Duplicate (listing_id, reviewer_id, date) triplets: 2

6. Unique identifier counts:
Unique review IDs (id): 84849
Unique listings with reviews (listing_id): 3191
Unique reviewers (reviewer_id): 75730


In [12]:
# Verification of duplicate triplets: compact aggregation and record preview
duplicate_triplets_all = reviews[reviews.duplicated(subset=['listing_id', 'reviewer_id', 'date'], keep=False)].sort_values(['listing_id', 'date'])

triplet_summary = duplicate_triplets_all.groupby(['listing_id', 'reviewer_id', 'date']).agg(
    row_count=('id', 'count'),
    unique_ids=('id', 'nunique'),
    unique_comments=('comments', 'nunique')
).reset_index()

print('1. Compact verification of same-day review triplets (unique IDs and texts):')
display(triplet_summary)

print('\n2. Record preview for the two triplet groups:')
display(duplicate_triplets_all[['listing_id', 'id', 'date', 'reviewer_id', 'comments']])


1. Compact verification of same-day review triplets (unique IDs and texts):


,listing_id,reviewer_id,date,row_count,unique_ids,unique_comments
0,447523,35607120,2015-07-06,2,2,2
1,3533224,39668556,2015-12-01,2,2,2



2. Record preview for the two triplet groups:


,listing_id,id,date,reviewer_id,comments
29086,447523,37438229,2015-07-06,35607120,The host canceled this reservation 47 days bef...
29087,447523,37438231,2015-07-06,35607120,The host canceled this reservation 53 days bef...
16387,3533224,55354340,2015-12-01,39668556,"Michael is a good host,He was not at home when..."
16388,3533224,55554854,2015-12-01,39668556,非常好


In [13]:
# 7-9. Temporal coverage, review counts per listing, and alignment with listings.csv
print('7. Temporal coverage:')
min_date = reviews['date'].min()
max_date = reviews['date'].max()
date_parse_errors = pd.to_datetime(reviews['date'], errors='coerce').isna().sum()
print(f'Earliest review date: {min_date}')
print(f'Latest review date: {max_date}')
print(f'Date parsing errors: {date_parse_errors}')

print('\n8. Distribution of reviews per listing:')
reviews_per_listing = reviews.groupby('listing_id').size()
summary_stats = pd.DataFrame({
    'metric': ['min', '25%', 'median', 'mean', '75%', 'max'],
    'value': [
        reviews_per_listing.min(),
        reviews_per_listing.quantile(0.25),
        reviews_per_listing.median(),
        round(reviews_per_listing.mean(), 2),
        reviews_per_listing.quantile(0.75),
        reviews_per_listing.max(),
    ]
})
display(summary_stats)

print('\n9. Alignment with listings.csv:')
listings_all = set(listings['id'])
listings_in_reviews = set(reviews['listing_id'])
listings_zero_reviews = set(listings.loc[listings['number_of_reviews'] == 0, 'id'])

print(f'Total listings in listings.csv: {len(listings_all)}')
print(f'Listings present in reviews.csv: {len(listings_in_reviews)}')
print(f'Listings with 0 reviews in listings.csv: {len(listings_zero_reviews)}')
print(f'All review listing_ids belong to listings.csv: {listings_in_reviews.issubset(listings_all)}')
print(f'Overlap between 0-review listings and reviews.csv: {len(listings_zero_reviews.intersection(listings_in_reviews))}')
print(f'Exact accounting check (3,191 + 627 == 3,818): {len(listings_in_reviews) + len(listings_zero_reviews) == len(listings_all)}')


7. Temporal coverage:
Earliest review date: 2009-06-07
Latest review date: 2016-01-03
Date parsing errors: 0

8. Distribution of reviews per listing:


,metric,value
0,min,1.00
1,25%,4.00
2,median,13.00
3,mean,26.59
4,75%,31.00
5,max,474.00



9. Alignment with listings.csv:
Total listings in listings.csv: 3818
Listings present in reviews.csv: 3191
Listings with 0 reviews in listings.csv: 627
All review listing_ids belong to listings.csv: True
Overlap between 0-review listings and reviews.csv: 0
Exact accounting check (3,191 + 627 == 3,818): True


#### 3.3.2 Preliminary Findings

All observations below are strictly verified by executed code on `reviews.csv`:

1. **Dataset shape and structure:** `reviews.csv` contains `84,849` rows and `6` columns (`listing_id`, `id`, `date`, `reviewer_id`, `reviewer_name`, `comments`).
2. **Data types:** Three integer identifiers (`listing_id`, `id`, `reviewer_id` are `int64`) and three text fields (`date`, `reviewer_name`, `comments` are `object`).
3. **Missing values:** `comments` has exactly `18` missing values (`0.02%`). All other five columns have `0` missing values (`0.00%`). No whitespace-only empty strings were found among non-null comments.
4. **Duplicates and uniqueness:**
   - `0` duplicate full rows.
   - `0` duplicate review `id`s (all `84,849` review IDs are strictly unique).
   - Exactly `2` duplicate `(listing_id, reviewer_id, date)` triplets exist. Aggregation confirms that both pairs have `unique_ids == 2` and `unique_comments == 2`, mathematically demonstrating distinct review IDs and distinct comment texts, while the underlying cause for same-day submissions remains unverified.
5. **Temporal coverage:** Review dates span from `2009-06-07` through `2016-01-03` (`0` date parsing errors), preceding the `2016-01-04` listings snapshot.
6. **Reviewer and listing participation:**
   - `84,849` reviews were written by `75,730` unique reviewers across `3,191` unique listings.
   - The number of reviews per listing ranges from `1` to `474` (median: `13.0`, mean: `26.59`).
7. **Cross-table alignment with `listings.csv`:**
   - `100%` of the `3,191` `listing_id`s in `reviews.csv` exist in `listings.csv` (`0` orphaned reviews).
   - Exactly `627` listings in `listings.csv` have no reviews in `reviews.csv`, exactly matching `listings['number_of_reviews'] == 0` (`3,191 + 627 = 3,818`).

##### Analytical Considerations for Next Steps
- **Row grain:** One record per individual guest review (review level).
- **Feature extraction candidates:** To merge review information into the listing-level table without expanding rows, review signals (e.g., total volume, recency, sentiment polarity, topic distributions) must be aggregated to `listing_id`.
- **Null handling:** Imputation or specialized indicators will be needed for the `627` listings with zero reviews to avoid dropping them during supervised modeling.


### 3.4 Walk Score — `seattle_walkscore_raw.csv`

#### 3.4.1 Data Exploration

In this section, we inspect `seattle_walkscore_raw.csv` to understand its metrics, coverage, distribution, and compatibility with the neighborhood taxonomy in `listings.csv`.


In [14]:
# 1-3. Load seattle_walkscore_raw.csv, inspect shape, column types, and first rows
walkscore_path = DATA_DIR / 'seattle_walkscore_raw.csv'
assert walkscore_path.is_file(), f'seattle_walkscore_raw.csv not found at {walkscore_path}'

walkscore = pd.read_csv(walkscore_path)

print('1. Dataset shape (rows, columns):', walkscore.shape)
print('\n2. Column names and data types:')
display(walkscore.dtypes.astype(str).rename('dtype').to_frame())

print('\n3. First 5 rows:')
display(walkscore.head())


1. Dataset shape (rows, columns): (80, 7)

2. Column names and data types:


,dtype
rank,int64
neighborhood,object
walk_score,int64
transit_score,int64
bike_score,int64
population,int64
walkscore_url,object



3. First 5 rows:


,rank,neighborhood,walk_score,transit_score,bike_score,population,walkscore_url
0,1,International District,98,100,83,3396,https://www.walkscore.com/WA/Seattle/Internati...
1,2,Belltown,98,97,76,14163,https://www.walkscore.com/WA/Seattle/Belltown
2,3,First Hill,98,98,79,9294,https://www.walkscore.com/WA/Seattle/First_Hill
3,4,Pioneer Square,97,100,78,4159,https://www.walkscore.com/WA/Seattle/Pioneer_S...
4,5,South Lake Union,94,85,80,4054,https://www.walkscore.com/WA/Seattle/South_Lak...


In [15]:
# 4-6. Missing values, duplicates, and metric distributions
print('4. Missing values (count and percentage):')
walkscore_missing = pd.DataFrame({
    'missing_count': walkscore.isna().sum(),
    'missing_pct': (walkscore.isna().mean() * 100).round(2),
})
display(walkscore_missing)

print('\n5. Duplicate checks:')
full_dupes = walkscore.duplicated().sum()
name_dupes = walkscore['neighborhood'].duplicated().sum()
print(f'Duplicate full rows: {full_dupes}')
print(f'Duplicate neighborhood names: {name_dupes}')
print(f'Unique neighborhood count: {walkscore["neighborhood"].nunique()}')

print('\n6. Summary statistics for walkability and transit scores:')
score_stats = walkscore[['walk_score', 'transit_score', 'bike_score', 'population']].describe().round(2)
display(score_stats)


4. Missing values (count and percentage):


,missing_count,missing_pct
rank,0,0.0
neighborhood,0,0.0
walk_score,0,0.0
transit_score,0,0.0
bike_score,0,0.0
population,0,0.0
walkscore_url,0,0.0



5. Duplicate checks:
Duplicate full rows: 0
Duplicate neighborhood names: 0
Unique neighborhood count: 80

6. Summary statistics for walkability and transit scores:


,walk_score,transit_score,bike_score,population
count,80.00,80.00,80.00,80.00
mean,70.04,57.70,67.84,7027.41
std,16.95,14.79,13.97,4677.78
min,22.00,33.00,23.00,529.00
25%,59.50,48.75,58.75,4475.00
50%,73.00,55.00,71.00,6066.50
75%,83.25,62.00,79.00,8045.75
max,98.00,100.00,89.00,29310.00


In [16]:
# 7. Cross-table alignment with listings.csv on neighborhood
listings_hoods = set(listings['neighbourhood_cleansed'].unique())
ws_hoods = set(walkscore['neighborhood'].unique())

exact_matches = listings_hoods.intersection(ws_hoods)
unmatched_in_listings = sorted(list(listings_hoods - ws_hoods))
unmatched_in_ws = sorted(list(ws_hoods - listings_hoods))

unmatched_listings_count = listings['neighbourhood_cleansed'].isin(unmatched_in_listings).sum()
unmatched_listings_pct = round(unmatched_listings_count / len(listings) * 100, 2)
matched_listings_count = len(listings) - unmatched_listings_count
matched_listings_pct = round(100.0 - unmatched_listings_pct, 2)

print(f'Unique neighborhoods in listings.csv (neighbourhood_cleansed): {len(listings_hoods)}')
print(f'Unique neighborhoods in seattle_walkscore_raw.csv (neighborhood): {len(ws_hoods)}')
print(f'Exact character-for-character matches: {len(exact_matches)}')
print(f'Neighborhoods in listings.csv without direct match: {len(unmatched_in_listings)}')
print(f'Neighborhoods in Walk Score not directly in listings.csv: {len(unmatched_in_ws)}')
print(f'\nListings covered by exact match: {matched_listings_count} ({matched_listings_pct}%)')
print(f'Listings in unmatched neighborhoods: {unmatched_listings_count} ({unmatched_listings_pct}%)')

print('\nListings neighborhoods requiring crosswalk mapping (14):')
for hood in unmatched_in_listings:
    count = (listings['neighbourhood_cleansed'] == hood).sum()
    print(f'  - {hood}: {count} listings')


Unique neighborhoods in listings.csv (neighbourhood_cleansed): 87
Unique neighborhoods in seattle_walkscore_raw.csv (neighborhood): 80
Exact character-for-character matches: 73
Neighborhoods in listings.csv without direct match: 14
Neighborhoods in Walk Score not directly in listings.csv: 7

Listings covered by exact match: 2733 (71.58%)
Listings in unmatched neighborhoods: 1085 (28.42%)

Listings neighborhoods requiring crosswalk mapping (14):
  - Adams: 70 listings
  - Alki: 42 listings
  - Broadway: 397 listings
  - Cedar Park: 8 listings
  - Central Business District: 103 listings
  - Harrison/Denny-Blaine: 14 listings
  - Holly Park: 6 listings
  - Mann: 67 listings
  - Minor: 135 listings
  - North Beach/Blue Ridge: 14 listings
  - Pike-Market: 28 listings
  - Stevens: 119 listings
  - Wedgwood: 18 listings
  - West Woodland: 64 listings


In [17]:
# 8. Top-ranked neighborhoods by mobility metric (accounting for ties)
print('Top neighborhoods by Walk Score (accounting for ties):')
top_walk = walkscore.nlargest(3, 'walk_score', keep='all')[['neighborhood', 'walk_score']].reset_index(drop=True)
display(top_walk)

print('\nTop neighborhoods by Transit Score (accounting for ties):')
top_transit = walkscore.nlargest(3, 'transit_score', keep='all')[['neighborhood', 'transit_score']].reset_index(drop=True)
display(top_transit)

print('\nTop neighborhoods by Bike Score (accounting for ties):')
top_bike = walkscore.nlargest(3, 'bike_score', keep='all')[['neighborhood', 'bike_score']].reset_index(drop=True)
display(top_bike)


Top neighborhoods by Walk Score (accounting for ties):


,neighborhood,walk_score
0,International District,98
1,Belltown,98
2,First Hill,98



Top neighborhoods by Transit Score (accounting for ties):


,neighborhood,transit_score
0,International District,100
1,Pioneer Square,100
2,First Hill,98
3,Yesler Terrace,98



Top neighborhoods by Bike Score (accounting for ties):


,neighborhood,bike_score
0,Roosevelt,89
1,Green Lake,88
2,University District,87


#### 3.4.2 Preliminary Findings

All observations below are strictly verified by executed code on `seattle_walkscore_raw.csv`:

1. **Dataset shape and structure:** `seattle_walkscore_raw.csv` contains `80` rows and `7` columns (`rank`, `neighborhood`, `walk_score`, `transit_score`, `bike_score`, `population`, `walkscore_url`).
2. **Data types:** Five integer metrics (`rank`, `walk_score`, `transit_score`, `bike_score`, `population` are `int64`) and two string columns (`neighborhood`, `walkscore_url` are `object`).
3. **Missing values and duplicates:**
   - `0` missing values across all `7` columns (`100%` complete).
   - `0` duplicate full rows.
   - Exactly `80` unique `neighborhood` values, establishing `neighborhood` as the primary key of this table.
4. **Metric distributions and confirmed top scores (80 neighborhoods):**
   - `walk_score`: ranges from `22` to `98` (median: `73.0`, mean: `70.04`). Top-ranked neighborhoods (tied at `98`): International District, Belltown, First Hill.
   - `transit_score`: ranges from `33` to `100` (median: `55.0`, mean: `57.70`). Top-ranked neighborhoods (tied at `100`): International District, Pioneer Square; followed by First Hill (`98`) and Yesler Terrace (`98`).
   - `bike_score`: ranges from `23` to `89` (median: `71.0`, mean: `67.84`). Top-ranked neighborhoods: Roosevelt (`89`), Green Lake (`88`), University District (`87`).
   - `population`: ranges from `529` to `29,310` (median: `6,066.5`, mean: `7,027.41`).
5. **Alignment with `listings.csv`:**
   - `listings.csv` contains `87` unique neighborhoods in `neighbourhood_cleansed`.
   - Exactly `73` neighborhoods match character-for-character between the two datasets, covering `2,733` listings (`71.58%`).
   - Exactly `14` neighborhoods in `listings.csv` do not have an exact string match in `seattle_walkscore_raw.csv`, representing `1,085` listings (`28.42%`): Adams (`70`), Alki (`42`), Broadway (`397`), Cedar Park (`8`), Central Business District (`103`), Harrison/Denny-Blaine (`14`), Holly Park (`6`), Mann (`67`), Minor (`135`), North Beach/Blue Ridge (`14`), Pike-Market (`28`), Stevens (`119`), Wedgwood (`18`), West Woodland (`64`).
   - Note: Character-matching alone does not confirm identical geographic boundaries between the two sources.

##### Analytical Considerations for Next Steps
- **Row grain:** One record per Seattle neighborhood (neighborhood level).
- **Join strategy evaluation:** A direct join leaves `28.42%` of listings unmapped. Candidate integration approaches (such as building a verified neighborhood crosswalk dictionary or utilizing `neighbourhood_group_cleansed` aggregation) should be evaluated during Data Preparation, subject to empirical verification of geographic boundaries.


### 3.5 Holidays & Events — `seattle_holidays_events_2016.csv`

#### 3.5.1 Data Exploration

In this section, we examine `seattle_holidays_events_2016.csv` to understand its event classifications, temporal coverage, geographic granularity, data completeness, and compatibility with the `calendar.csv` daily timeline.


In [18]:
# 1-3. Load seattle_holidays_events_2016.csv, inspect shape, column types, and complete table
holidays_path = DATA_DIR / 'seattle_holidays_events_2016.csv'
assert holidays_path.is_file(), f'seattle_holidays_events_2016.csv not found at {holidays_path}'

holidays = pd.read_csv(holidays_path)

print('1. Dataset shape (rows, columns):', holidays.shape)
print('\n2. Column names and data types:')
display(holidays.dtypes.to_frame(name='dtype'))

print('\n3. Complete dataset (all 19 records):')
display(holidays)


1. Dataset shape (rows, columns): (19, 5)

2. Column names and data types:


,dtype
date,object
event_name,object
type,object
neighborhood_focus,object
source,object



3. Complete dataset (all 19 records):


,date,event_name,type,neighborhood_focus,source
0,2016-01-01,New Year's Day,Federal Holiday,Citywide,U.S. OPM
1,2016-01-18,Martin Luther King Jr. Day,Federal Holiday,Citywide,U.S. OPM
2,2016-02-15,Washington's Birthday (Presidents' Day),Federal Holiday,Citywide,U.S. OPM
3,2016-05-30,Memorial Day,Federal Holiday,Citywide,U.S. OPM
4,2016-06-25,Seattle PrideFest,Seattle Event,Capitol Hill,Seattle Pride / Seattle Times Archive
5,2016-06-26,Seattle Pride Parade,Seattle Event,Downtown / Capitol Hill,Seattle Pride / Seattle Times Archive
6,2016-07-04,Independence Day,Federal Holiday,Citywide,U.S. OPM
7,2016-08-05,Seafair Weekend,Seattle Festival,Lake Washington / Rainier Valley,Seafair Archive
8,2016-08-06,Seafair Weekend & Blue Angels,Seattle Festival,Lake Washington / Rainier Valley,Seafair Archive
9,2016-08-07,Seafair Weekend & Blue Angels,Seattle Festival,Lake Washington / Rainier Valley,Seafair Archive


In [19]:
# 4-6. Missing values, duplicates, date validation, event accounting, and groupings
print('4. Missing values (count and percentage):')
holidays_missing = pd.DataFrame({
    'missing_count': holidays.isna().sum(),
    'missing_pct': (holidays.isna().mean() * 100).round(2),
})
display(holidays_missing)

print('\n5. Duplicate checks and date parsing:')
full_dupes = holidays.duplicated().sum()
date_dupes = holidays['date'].duplicated().sum()
print(f'Full duplicate rows: {full_dupes}')
print(f'Duplicate dates: {date_dupes}')
print(f'Unique dates: {holidays["date"].nunique()}')

parsed_dates = pd.to_datetime(holidays['date'], format='%Y-%m-%d', errors='coerce')
print(f'Date parse errors: {parsed_dates.isna().sum()}')
print(f'Date range: {parsed_dates.min().strftime("%Y-%m-%d")} to {parsed_dates.max().strftime("%Y-%m-%d")}')
print(f'Dates strictly sorted ascending: {parsed_dates.is_monotonic_increasing}')

print('\n6. Event type accounting and multi-day groupings:')
is_fed = holidays['type'].str.contains('Federal Holiday')
is_evt = holidays['type'].str.contains('Festival|Event|Convention')
overlap = holidays[is_fed & is_evt]
print(f'Federal holiday dates: {is_fed.sum()}')
print(f'Local festival / convention / event dates: {is_evt.sum()}')
print(f'Joint federal holiday & convention dates: {len(overlap)} ({overlap["date"].iloc[0]}: {overlap["event_name"].iloc[0]})')
print(f'Total unique dates: {is_fed.sum()} + {is_evt.sum()} - {len(overlap)} = {len(holidays)}')

# Programmatic classification of multi-day event series
def classify_event_series(event_name):
    if 'Pride' in event_name:
        return 'Seattle Pride'
    elif 'Seafair' in event_name:
        return 'Seafair Weekend'
    elif 'PAX' in event_name or 'Bumbershoot' in event_name:
        return 'PAX West / Bumbershoot'
    return None

holidays_work = holidays.copy()
holidays_work['event_series'] = holidays_work['event_name'].apply(classify_event_series)
multi_day_df = holidays_work.dropna(subset=['event_series'])

multi_day_summary = (
    multi_day_df.groupby('event_series')
    .agg(
        unique_dates=('date', 'nunique'),
        start_date=('date', 'min'),
        end_date=('date', 'max'),
        neighborhood_foci=('neighborhood_focus', lambda s: sorted(list(s.unique()))),
        event_names=('event_name', lambda s: list(s.unique())),
    )
    .reset_index()
)
print('\nMulti-day event groups (dynamically aggregated):')
display(multi_day_summary)

print('\n7. Geographic focus and source attribution distributions:')
print('By neighborhood focus (counts and percentage):')
focus_dist = holidays['neighborhood_focus'].value_counts(dropna=False).to_frame(name='count')
focus_dist['percentage'] = (focus_dist['count'] / len(holidays) * 100).round(2)
display(focus_dist)

print('\nBy data source (as reported in source):')
display(holidays['source'].value_counts(dropna=False).to_frame(name='count'))


4. Missing values (count and percentage):


,missing_count,missing_pct
date,0,0.0
event_name,0,0.0
type,0,0.0
neighborhood_focus,0,0.0
source,0,0.0



5. Duplicate checks and date parsing:
Full duplicate rows: 0
Duplicate dates: 0
Unique dates: 19
Date parse errors: 0
Date range: 2016-01-01 to 2017-01-02
Dates strictly sorted ascending: True

6. Event type accounting and multi-day groupings:
Federal holiday dates: 11
Local festival / convention / event dates: 9
Joint federal holiday & convention dates: 1 (2016-09-05: Labor Day / PAX West)
Total unique dates: 11 + 9 - 1 = 19

Multi-day event groups (dynamically aggregated):


,event_series,unique_dates,start_date,end_date,neighborhood_foci,event_names
0,PAX West / Bumbershoot,4,2016-09-02,2016-09-05,"[Downtown / Citywide, Downtown / Seattle Center]","[PAX West / Bumbershoot, Labor Day / PAX West]"
1,Seafair Weekend,3,2016-08-05,2016-08-07,[Lake Washington / Rainier Valley],"[Seafair Weekend, Seafair Weekend & Blue Angels]"
2,Seattle Pride,2,2016-06-25,2016-06-26,"[Capitol Hill, Downtown / Capitol Hill]","[Seattle PrideFest, Seattle Pride Parade]"



7. Geographic focus and source attribution distributions:
By neighborhood focus (counts and percentage):


,count,percentage
neighborhood_focus,,
Citywide,10,52.63
Lake Washington / Rainier Valley,3,15.79
Downtown / Seattle Center,3,15.79
Capitol Hill,1,5.26
Downtown / Capitol Hill,1,5.26
Downtown / Citywide,1,5.26



By data source (as reported in source):


,count
source,
U.S. OPM,10
Seafair Archive,3
Penny Arcade / Bumbershoot Archives,3
Seattle Pride / Seattle Times Archive,2
U.S. OPM / Penny Arcade Archive,1


In [20]:
# 7. Temporal alignment with calendar.csv and spatial alignment with listings.csv
cal_dates = set(calendar['date'].unique())
hol_dates = set(holidays['date'].unique())

matched_dates = sorted(list(hol_dates.intersection(cal_dates)))
unmatched_dates = sorted(list(hol_dates - cal_dates))
unlisted_cal_dates = sorted(list(cal_dates - hol_dates))

print('8. Temporal alignment with calendar.csv:')
print(f'Calendar date span: {len(cal_dates)} days ({min(cal_dates)} to {max(cal_dates)})')
print(f'Holiday dates within calendar window: {len(matched_dates)} / {len(hol_dates)} ({len(matched_dates)/len(hol_dates)*100:.2f}%)')
print(f'Calendar dates with no event recorded in this source: {len(unlisted_cal_dates)} / {len(cal_dates)} ({len(unlisted_cal_dates)/len(cal_dates)*100:.2f}%)')
print(f'Holiday dates outside calendar window: {len(unmatched_dates)}')

if unmatched_dates:
    print('\nUnmatched holiday date details:')
    display(holidays[holidays['date'].isin(unmatched_dates)][['date', 'event_name', 'type', 'source']])

cal_matched_rows = calendar[calendar['date'].isin(matched_dates)]
total_cal_rows = len(calendar)
matched_rows_count = len(cal_matched_rows)
pct_matched = (matched_rows_count / total_cal_rows) * 100

listings_per_matched_date = cal_matched_rows.groupby('date')['listing_id'].nunique()
total_listings = len(listings)
min_listings = listings_per_matched_date.min()
max_listings = listings_per_matched_date.max()
cov_pct = (min_listings / total_listings) * 100

print(f'\nCalendar records matching holiday dates: {matched_rows_count:,} / {total_cal_rows:,} ({pct_matched:.2f}%)')
print(f'Listings per matched date: min = {min_listings:,}, max = {max_listings:,}')
print(f'Listing coverage on each matched date: {min_listings:,} / {total_listings:,} ({cov_pct:.2f}% coverage of listings in our dataset)')

print('\n9. Spatial alignment with listings.csv taxonomy:')
focus_vals = set(holidays['neighborhood_focus'].unique())
cleansed_hoods = set(listings['neighbourhood_cleansed'].unique())
group_hoods = set(listings['neighbourhood_group_cleansed'].unique())

match_cleansed = sorted(list(focus_vals.intersection(cleansed_hoods)))
match_groups = sorted(list(focus_vals.intersection(group_hoods)))

print(f'Unique neighborhood_focus values ({len(focus_vals)}): {sorted(list(focus_vals))}')
print(f'Exact string matches with neighbourhood_cleansed ({len(cleansed_hoods)} neighborhoods): {match_cleansed}')
print(f'Exact string matches with neighbourhood_group_cleansed ({len(group_hoods)} groups): {match_groups}')


8. Temporal alignment with calendar.csv:
Calendar date span: 365 days (2016-01-04 to 2017-01-02)
Holiday dates within calendar window: 18 / 19 (94.74%)
Calendar dates with no event recorded in this source: 347 / 365 (95.07%)
Holiday dates outside calendar window: 1

Unmatched holiday date details:


,date,event_name,type,source
0,2016-01-01,New Year's Day,Federal Holiday,U.S. OPM



Calendar records matching holiday dates: 68,724 / 1,393,570 (4.93%)
Listings per matched date: min = 3,818, max = 3,818
Listing coverage on each matched date: 3,818 / 3,818 (100.00% coverage of listings in our dataset)

9. Spatial alignment with listings.csv taxonomy:
Unique neighborhood_focus values (6): ['Capitol Hill', 'Citywide', 'Downtown / Capitol Hill', 'Downtown / Citywide', 'Downtown / Seattle Center', 'Lake Washington / Rainier Valley']
Exact string matches with neighbourhood_cleansed (87 neighborhoods): []
Exact string matches with neighbourhood_group_cleansed (17 groups): ['Capitol Hill']


#### 3.5.2 Preliminary Findings

All observations below are strictly verified by executed code on `seattle_holidays_events_2016.csv` and cross-table validation with `calendar.csv` and `listings.csv`:

1. **Dataset shape and structure:** `seattle_holidays_events_2016.csv` contains `19` rows and `5` columns (`date`, `event_name`, `type`, `neighborhood_focus`, `source`).
2. **Data types:** All `5` columns are string (`object`).
3. **Data completeness and uniqueness:**
   - `0` missing values across all columns (`100%` completeness within this file).
   - `0` duplicate full rows and `0` duplicate dates.
   - Exactly `19` unique `date` values, establishing `date` as the primary key of this table.
4. **Temporal coverage and date validation:**
   - Date strings parse strictly with `format='%Y-%m-%d'` with `0` errors.
   - Dates span from `2016-01-01` to `2017-01-02` and are strictly ordered ascending (`is_monotonic_increasing = True`).
5. **Event taxonomy and groupings:**
   - `11` federal holiday dates (10 purely federal holidays + 1 joint holiday/convention: `2016-09-05` Labor Day / PAX West).
   - `9` local festival, convention, or event dates (2 Pride, 3 Seafair, 4 PAX West / Bumbershoot, with Labor Day overlapping on `2016-09-05`).
   - Dynamic grouping shows three major multi-day clusters: Seattle Pride (2 days: June 25–26), Seafair Weekend (3 days: August 5–7), and PAX West / Bumbershoot (4 days: September 2–5).
6. **Source attribution:**
   - The `source` column documents origin metadata for all 19 rows: U.S. OPM (`10`), Seafair Archive (`3`), Penny Arcade / Bumbershoot Archives (`3`), Seattle Pride / Seattle Times Archive (`2`), and U.S. OPM / Penny Arcade Archive (`1`).
   - Note: This confirms attribution data entry completeness within this CSV, but does not constitute independent verification against primary historical archives or guarantee an exhaustive census of all 2016 Seattle events.
7. **Temporal alignment with `calendar.csv`:**
   - `calendar.csv` covers `365` days from `2016-01-04` to `2017-01-02`.
   - Exactly `18` of the `19` dates (`94.74%`) fall within calendar coverage, touching `68,724` calendar rows (`4.93%` of `1,393,570` total calendar observations).
   - Exactly `1` date (`2016-01-01`, New Year's Day) precedes the calendar start date (`2016-01-04`) and cannot be linked to calendar records.
   - Exactly `347` calendar dates have no event recorded in this source (`95.07%` of the 365 calendar days).
   - On each of the `18` matched dates, exactly `3,818` listings are present (`100.00%` coverage of listings in our dataset).
8. **Spatial alignment with `listings.csv`:**
   - `neighborhood_focus` contains `6` distinct descriptive labels (`Citywide`: 10, `Lake Washington / Rainier Valley`: 3, `Downtown / Seattle Center`: 3, `Capitol Hill`: 1, `Downtown / Capitol Hill`: 1, `Downtown / Citywide`: 1).
   - Exactly `0` match `listings['neighbourhood_cleansed']` (87 categories).
   - Exactly `1` label (`Capitol Hill`) matches `listings['neighbourhood_group_cleansed']` (17 categories).
   - String equality for `Capitol Hill` does not confirm identical geographic polygon boundaries between the event source and administrative neighborhood groups, and the other 5 values do not match either classification directly.

##### Analytical Considerations for Next Steps
- **Row grain:** One record per calendar date (daily date grain).
- **Calendar join strategy:** A left join from `calendar.csv` on `date` will incorporate event flags for the 18 matching dates while naturally assigning missing/unlisted status to the remaining 347 calendar dates. The unmapped date `2016-01-01` is naturally excluded as out-of-scope.
- **Handling unlisted dates:** Dates without a match in this table represent dates with no event recorded in this source; they should be treated as unlisted in this dataset, not as confirmed event-free days.
- **Geographic interaction:** Because most events are designated as `Citywide` or compound areas (e.g., `Downtown / Seattle Center`), spatial differentiation of event effects requires either mapping focus areas to `neighbourhood_group_cleansed` or evaluating citywide indicator flags during Feature Engineering.
- **Non-causality caution:** The relationship between listed events, price variations, and availability proxy remains to be empirically evaluated during EDA; the presence of an event on a given date does not in itself establish an impact on listing demand.
